# Module MLflow — Enregistrement au Model Registry

Ce notebook récupère la meilleure run MLflow (accuracy maximale) et enregistre le modèle au **Model Registry** sous le nom `EcoSmartClassifier`.

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient

# Connexion au tracking server local
mlflow.set_tracking_uri('sqlite:///mlflow.db')
client = MlflowClient()

EXPERIMENT_NAME = 'Waste_Classification_Multimodal'
MODEL_REGISTRY_NAME = 'EcoSmartClassifier'

experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    raise RuntimeError(f'Expérience {EXPERIMENT_NAME!r} introuvable. Lance d abord le notebook 06.')

print(f'Experiment ID : {experiment.experiment_id}')


In [ ]:
# Récupérer toutes les runs et trouver la meilleure accuracy
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=['metrics.accuracy DESC'],
    max_results=10
)

if not runs:
    raise RuntimeError('Aucune run trouvée.')

best_run = runs[0]
best_run_id = best_run.info.run_id
best_accuracy = best_run.data.metrics.get('accuracy', 0)

print(f'Meilleure run : {best_run_id}')
print(f'Accuracy      : {best_accuracy:.4f}')
print(f'Params        : {best_run.data.params}')


In [ ]:
# Enregistrement au Model Registry
model_uri = f'runs:/{best_run_id}/model'

registered = mlflow.register_model(
    model_uri=model_uri,
    name=MODEL_REGISTRY_NAME
)

print(f'Modèle enregistré : {registered.name} v{registered.version}')
print(f'Status            : {registered.status}')


In [ ]:
# Transition vers le stage 'Production'
from mlflow.tracking import MlflowClient

client.transition_model_version_stage(
    name=MODEL_REGISTRY_NAME,
    version=registered.version,
    stage='Production',
    archive_existing_versions=True
)

print(f'Modèle {MODEL_REGISTRY_NAME} v{registered.version} → Production ✅')


In [ ]:
# Vérification : lister tous les modèles en Production
prod_versions = client.get_latest_versions(MODEL_REGISTRY_NAME, stages=['Production'])
for v in prod_versions:
    print(f'  Version {v.version} | Run {v.run_id} | Stage: {v.current_stage}')
